# 05 - Modelado baseline (scikit-learn)

Fase **Modeling** de CRISP-DM. Entrenamos tres modelos clasicos de clasificacion binaria sobre `dataset_integrado.csv` para predecir `occupied` (1 = dia reservado / bloqueado, 0 = libre).

- **Logistic Regression** - referencia lineal rapida, con escalado.
- **Random Forest** - no lineal, robusto, feature importance interpretable.
- **Gradient Boosting** - normalmente el mas fuerte de los tres.

**Split por fecha** (no aleatorio) para simular prediccion: primeros 42 dias como train, ultimos 14 como test.

**Anti-leakage:** excluimos las columnas `availability_30/60/90/365` y `estimated_occupancy_l365d` porque son proxies directos del target (vienen del propio `calendar.csv`). Sin este filtro el modelo alcanza AUC ~0.94 pero por razones trampa; con el filtro se ven metricas honestas.

**Metricas objetivo del curso:** F1 >= 0.75, AUC >= 0.80.


## 0. Imports y carga

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, roc_curve)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

PROC_DIR = Path('../data/processed')
df = pd.read_csv(PROC_DIR / 'dataset_integrado.csv', parse_dates=['date'])
print(f'Dataset: {df.shape}')
print(f'Rango de fechas: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'% ocupado: {df["occupied"].mean()*100:.2f}%')
df.head()

---
## 1. Feature engineering ligero

No tocamos la lógica del dataset, solo codificamos las categóricas:
- Columnas con pocas categorías (`room_type`, `property_type`, `weather_cat`, `neighbourhood_group_cleansed`) → *one-hot encoding*.
- `neighbourhood_cleansed` (~70 valores) → *target mean encoding* calculado SOLO sobre el train para no filtrar información.

In [ ]:
# Columnas a eliminar del feature set
# (1) identificadores / redundantes
# (2) LEAKAGE: availability_* y estimated_occupancy_l365d vienen del propio
#     calendar.csv y son proxies directos del target -> hay que excluirlas
#     para que el modelo aprenda de verdad y no haga trampa.
drop_cols = [
    # identificadores
    'listing_id', 'date',
    'minimum_nights_day', 'maximum_nights_day',
    # anti-leakage
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
    'estimated_occupancy_l365d',
]
drop_cols = [c for c in drop_cols if c in df.columns]

# One-hot de las categoricas de baja cardinalidad
onehot_cols = ['room_type', 'property_type', 'weather_cat', 'neighbourhood_group_cleansed']
df_enc = pd.get_dummies(df.drop(columns=drop_cols),
                        columns=onehot_cols, drop_first=True)

# Pasar columnas bool de one-hot a int para que sklearn no refunfune
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

print(f'Shape tras one-hot: {df_enc.shape}')
print(f'Columnas finales:   {len(df_enc.columns)}')
print(f'Columnas eliminadas por leakage o ruido: {drop_cols}')


---
## 2. Split train / test por fecha

Dividimos por fecha para simular prediccion (no split aleatorio): primeros 42 dias como train, ultimos 14 como test.

Ademas calculamos **dos target encodings** usando solo el train:

- `neigh_enc` — media de ocupacion por barrio. Captura efecto *zona turistica*.
- `listing_te` — media de ocupacion por listing individual. Captura la popularidad intrinseca del piso, que en la practica es el mejor predictor de su ocupacion futura.

**Limitacion metodologica:** `listing_te` solo funciona para listings con historial en train. Para un listing completamente nuevo caeria a la media global. En este dataset los 18.172 listings aparecen tanto en train como en test, asi que la feature es plenamente utilizable. En un escenario real con *cold start* habria que complementarla con features de contenido (caracteristicas del piso, barrio, reviews, etc.).


In [ ]:
# Horizonte: 2025-12-14 -> 2026-02-07 (56 dias)
# Train: primeros 42 dias  (75%)
# Test : ultimos  14 dias  (25%)
SPLIT_DATE = pd.Timestamp('2026-01-25')   # 42 dias despues del scrape

mask_train = df['date'] < SPLIT_DATE
mask_test  = df['date'] >= SPLIT_DATE

X_cols = [c for c in df_enc.columns if c not in ('occupied',)]
y = df_enc['occupied']

X_train = df_enc.loc[mask_train, X_cols].copy()
y_train = y[mask_train]
X_test  = df_enc.loc[mask_test,  X_cols].copy()
y_test  = y[mask_test]

global_mean = y_train.mean()

# Target encoding por barrio (solo con train)
barrio_map = df.loc[mask_train].groupby('neighbourhood_cleansed')['occupied'].mean()
X_train['neigh_enc'] = df.loc[mask_train, 'neighbourhood_cleansed'].map(barrio_map).values
X_test['neigh_enc']  = df.loc[mask_test,  'neighbourhood_cleansed'].map(barrio_map).fillna(global_mean).values

# Target encoding por listing (solo con train)
# Captura popularidad intrinseca del piso. Imputamos con la media global para
# listings no vistos (en este dataset no pasa, pero lo dejamos defensivo).
listing_map = df.loc[mask_train].groupby('listing_id')['occupied'].mean()
X_train['listing_te'] = df.loc[mask_train, 'listing_id'].map(listing_map).fillna(global_mean).values
X_test['listing_te']  = df.loc[mask_test,  'listing_id'].map(listing_map).fillna(global_mean).values

if 'neighbourhood_cleansed' in X_train.columns:
    X_train = X_train.drop(columns=['neighbourhood_cleansed'])
    X_test  = X_test.drop(columns=['neighbourhood_cleansed'])

# Sanity check: listings no vistos en test
listings_train = set(df.loc[mask_train, 'listing_id'].unique())
listings_test  = set(df.loc[mask_test,  'listing_id'].unique())
nuevos = listings_test - listings_train
print(f'Train: {X_train.shape} - % ocupado {y_train.mean()*100:.1f}%')
print(f'Test : {X_test.shape} - % ocupado {y_test.mean()*100:.1f}%')
print(f'Fecha split: {SPLIT_DATE.date()}')
print(f'Listings sin historial en test (cold start): {len(nuevos)}')


---
## 3. Logistic Regression (baseline lineal)

Requiere escalado de variables numéricas.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
logreg.fit(X_train_sc, y_train)

y_pred_lr  = logreg.predict(X_test_sc)
y_proba_lr = logreg.predict_proba(X_test_sc)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr  = f1_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)

print(f'Accuracy : {acc_lr:.4f}')
print(f'F1       : {f1_lr:.4f}')
print(f'AUC      : {auc_lr:.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['libre','ocupado']))

---
## 4. Random Forest

Bosque de 200 arboles, profundidad maxima 15 y al menos 5 muestras por hoja. Sin `class_weight='balanced'` a proposito: la clase *ocupada* pesa 55.7% del dataset, es un desbalanceo leve y forzar balanceo penalizaba el F1 de la mayoritaria. Dejamos que el modelo aprenda la distribucion real.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = f1_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)

print(f'Accuracy : {acc_rf:.4f}')
print(f'F1       : {f1_rf:.4f}')
print(f'AUC      : {auc_rf:.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=['libre','ocupado']))


---
## 5. Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=42
)
gb.fit(X_train, y_train)

y_pred_gb  = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

acc_gb = accuracy_score(y_test, y_pred_gb)
f1_gb  = f1_score(y_test, y_pred_gb)
auc_gb = roc_auc_score(y_test, y_proba_gb)

print(f'Accuracy : {acc_gb:.4f}')
print(f'F1       : {f1_gb:.4f}')
print(f'AUC      : {auc_gb:.4f}')
print()
print(classification_report(y_test, y_pred_gb, target_names=['libre','ocupado']))

---
## 6. Comparativa de modelos

In [ ]:
resultados = pd.DataFrame({
    'modelo':   ['Logistic Regression','Random Forest','Gradient Boosting'],
    'accuracy': [acc_lr, acc_rf, acc_gb],
    'f1':       [f1_lr,  f1_rf,  f1_gb],
    'auc':      [auc_lr, auc_rf, auc_gb],
}).round(4).sort_values('auc', ascending=False)

print(resultados.to_string(index=False))

# Objetivo del proyecto
print('\n--- Objetivos ---')
for _, r in resultados.iterrows():
    ok_f1  = '✓' if r['f1']  >= 0.75 else '✗'
    ok_auc = '✓' if r['auc'] >= 0.80 else '✗'
    print(f'{r["modelo"]:22s}  F1>=0.75 {ok_f1}   AUC>=0.80 {ok_auc}')

In [ ]:
# Gráfico comparativo de métricas
fig, ax = plt.subplots(figsize=(9, 4))
resultados_plot = resultados.melt(id_vars='modelo', value_vars=['accuracy','f1','auc'],
                                  var_name='métrica', value_name='valor')
sns.barplot(data=resultados_plot, x='modelo', y='valor', hue='métrica', ax=ax)
ax.set_ylim(0, 1)
ax.set_title('Comparativa de métricas por modelo')
ax.axhline(0.75, ls='--', color='gray', alpha=0.5, label='umbral F1 0.75')
ax.axhline(0.80, ls=':',  color='gray', alpha=0.5, label='umbral AUC 0.80')
plt.tight_layout()
plt.show()

---
## 7. Curvas ROC y matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
for nombre, y_proba, color in [
    ('LogReg',  y_proba_lr, 'steelblue'),
    ('RF',      y_proba_rf, 'darkorange'),
    ('GradBoost', y_proba_gb, 'seagreen')
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, color=color, label=f'{nombre}  (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1], ls='--', color='gray')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('Curvas ROC')
axes[0].legend()

# Matriz de confusión del mejor modelo
mejor = resultados.iloc[0]['modelo']
y_pred_mejor = {'Logistic Regression': y_pred_lr,
                'Random Forest': y_pred_rf,
                'Gradient Boosting': y_pred_gb}[mejor]
cm = confusion_matrix(y_test, y_pred_mejor)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['libre','ocupado'], yticklabels=['libre','ocupado'],
            ax=axes[1], cbar=False)
axes[1].set_title(f'Matriz de confusión — {mejor}')
axes[1].set_xlabel('Predicho')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

---
## 8. Importancia de variables (Random Forest)

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X_train.columns) \
        .sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=imp.values, y=imp.index, ax=ax, color='steelblue')
ax.set_title('Top 20 variables más importantes — Random Forest')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

print(imp.round(4).to_string())

---
## 9. Optimización del threshold (Random Forest)

El F1-score por defecto se calcula con umbral 0.5 (clasificamos como *ocupado* si `p(occupied) >= 0.5`).
Ese umbral es arbitrario y casi nunca es óptimo para F1. Buscamos el mejor threshold **en train** barriendo un rango, y lo aplicamos en test para re-calcular métricas sin hacer data snooping.

Solo optimizamos el modelo ganador por AUC (Random Forest) para no inflar las métricas del resto.


In [ ]:
from sklearn.metrics import precision_recall_curve

# 1. Probabilidades del RF sobre train (para buscar el threshold)
y_train_proba = rf.predict_proba(X_train)[:, 1]

# 2. Barrido con precision_recall_curve (thresholds ordenados por sklearn)
prec, rec, thr = precision_recall_curve(y_train, y_train_proba)
# F1 por cada threshold (evitamos division por cero)
f1_scores = 2 * prec * rec / (prec + rec + 1e-12)

# precision_recall_curve devuelve len(thr) = len(prec)-1, alineamos
best_idx = f1_scores[:-1].argmax()
best_thr = thr[best_idx]
print(f'Threshold optimo (maximiza F1 en TRAIN): {best_thr:.4f}')
print(f'F1 en TRAIN con ese threshold:          {f1_scores[best_idx]:.4f}')

# 3. Aplicamos el threshold en TEST y re-calculamos metricas
y_test_proba = rf.predict_proba(X_test)[:, 1]
y_pred_opt = (y_test_proba >= best_thr).astype(int)

acc_opt = accuracy_score(y_test, y_pred_opt)
f1_opt  = f1_score(y_test, y_pred_opt)
auc_opt = roc_auc_score(y_test, y_test_proba)  # AUC no depende del threshold

print()
print('--- Random Forest: default vs threshold optimo ---')
print(f"{'Metrica':<12} {'thr=0.5':>10} {'thr=' + f'{best_thr:.3f}':>12}")
print(f"{'Accuracy':<12} {accuracy_score(y_test, rf.predict(X_test)):>10.4f} {acc_opt:>12.4f}")
print(f"{'F1':<12} {f1_score(y_test, rf.predict(X_test)):>10.4f} {f1_opt:>12.4f}")
print(f"{'AUC':<12} {roc_auc_score(y_test, y_test_proba):>10.4f} {auc_opt:>12.4f}")

# 4. Comprobamos objetivos del curso
cumple_f1  = 'OK' if f1_opt  >= 0.75 else 'NO'
cumple_auc = 'OK' if auc_opt >= 0.80 else 'NO'
print()
print(f'Objetivo F1  >= 0.75 : {cumple_f1}  (actual {f1_opt:.4f})')
print(f'Objetivo AUC >= 0.80 : {cumple_auc}  (actual {auc_opt:.4f})')

# 5. Visualizacion del trade-off precision-recall-F1 vs threshold
import numpy as np
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thr, prec[:-1], label='Precision', lw=2)
ax.plot(thr, rec[:-1],  label='Recall',    lw=2)
ax.plot(thr, f1_scores[:-1], label='F1', lw=2, color='crimson')
ax.axvline(best_thr, color='gray', linestyle='--', alpha=0.7, label=f'thr optimo = {best_thr:.3f}')
ax.axvline(0.5, color='black', linestyle=':', alpha=0.5, label='thr default = 0.5')
ax.set_xlabel('Threshold')
ax.set_ylabel('Metrica')
ax.set_title('Precision / Recall / F1 vs Threshold (Random Forest, TRAIN)')
ax.legend(loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
## 10. Conclusiones baseline

1. **Modelo ganador:** el que tenga mayor AUC en la tabla de la sección 6 (normalmente **Gradient Boosting** o **Random Forest**).
2. **Cumplimiento de objetivos:** si F1 ≥ 0.75 y AUC ≥ 0.80 → se cumplen los umbrales definidos en la entrega 1.
3. **Variables más predictivas:** `estimated_occupancy_l365d`, `availability_30/60/90`, `neigh_enc` (barrio codificado) y `reviews_per_month` suelen dominar la importancia. Esto indica que la ocupación histórica del propio anuncio es la señal más fuerte — coherente con lo esperado.
4. **Clima y día de la semana** aportan menos de lo que inicialmente imaginábamos porque en el horizonte corto de 8 semanas la variabilidad es limitada.

### Siguiente paso
- Subir `dataset_integrado.csv` a **BigQuery** y replicar el pipeline en **Orange Data Mining** para la entrega visual.
- El dashboard final en **Power BI** consumirá las tablas de BigQuery.
- Opcional: afinar hiperparámetros con `GridSearchCV` en el mejor de los tres modelos.